In [0]:
%pip install yfinance

In [0]:
import yfinance as yf
import json
import time
import random
import os
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# -----------------------------
# BASE PATH (Unity Catalog Volume)
# -----------------------------
BASE_PATH = "/Volumes/company-risk-intelligence-platform/bronze/raw_data/yfinance/"

SUBFOLDERS = [
    "stock",
    "news",
    "financials",
    "balance_sheet",
    "cashflow",
    "info"
]

# ensure folders exist
for folder in SUBFOLDERS:
    os.makedirs(BASE_PATH + folder, exist_ok=True)


# -----------------------------
# COMPANIES
# -----------------------------
companies = [
    {"name": "J SAINSBURY PLC", "ticker": "SBRY.L"},
    {"name": "JD SPORTS FASHION PLC", "ticker": "JD.L"},
    {"name": "OCADO GROUP PLC", "ticker": "OCDO.L"},
    {"name": "LLOYDS BANKING GROUP PLC", "ticker": "LLOY.L"},
    {"name": "NATWEST GROUP PLC", "ticker": "NWG.L"},
    {"name": "NATIONAL GRID PLC", "ticker": "NG.L"},
    {"name": "SSE PLC", "ticker": "SSE.L"},
    {"name": "DRAX GROUP PLC", "ticker": "DRX.L"},
    {"name": "BALFOUR BEATTY PLC", "ticker": "BBY.L"},
    {"name": "PERSIMMON PLC", "ticker": "PSN.L"},
    {"name": "EASYJET PLC", "ticker": "EZJ.L"},
    {"name": "INTERCONTINENTAL HOTELS GROUP PLC", "ticker": "IHG.L"},
    {"name": "HIKMA PHARMACEUTICALS PLC", "ticker": "HIK.L"},
    {"name": "BT GROUP PLC", "ticker": "BT-A.L"},
    {"name": "ITV PLC", "ticker": "ITV.L"},
    {"name": "JOHNSON MATTHEY PLC", "ticker": "JMAT.L"},
    {"name": "MELROSE INDUSTRIES PLC", "ticker": "MRO.L"},
    {"name": "AVIVA PLC", "ticker": "AV.L"},
    {"name": "ASTON MARTIN LAGONDA GLOBAL HOLDINGS PLC", "ticker": "AML.L"},
]


# -----------------------------
# SAFE SAVE FUNCTION
# -----------------------------
def save_json(folder, filename, data):
    path = f"{BASE_PATH}{folder}/{filename}"
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    return path


# -----------------------------
# INGEST FUNCTION
# -----------------------------
def ingest_company(company):

    ticker = company["ticker"]
    name = company["name"]
    safe_ticker = ticker.replace(".", "_")

    print(f"\nProcessing {name} ({ticker})")

    try:
        stock = yf.Ticker(ticker)

        # ---------------- STOCK ----------------
        try:
            hist = stock.history(period="1y")
            if not hist.empty:
                save_json(
                    "stock",
                    f"{safe_ticker}.json",
                    hist.reset_index().to_dict(orient="records")
                )
        except Exception as e:
            print(f"stock error {ticker}: {e}")

        # ---------------- NEWS ----------------
        try:
            news = stock.news
            if news:
                save_json("news", f"{safe_ticker}.json", news)
        except Exception as e:
            print(f"news error {ticker}: {e}")

        # ---------------- FINANCIALS ----------------
        try:
            fin = stock.financials
            if fin is not None and not fin.empty:
                save_json(
                    "financials",
                    f"{safe_ticker}.json",
                    fin.reset_index().to_dict(orient="records")
                )
        except Exception as e:
            print(f"financials error {ticker}: {e}")

        # ---------------- BALANCE SHEET ----------------
        try:
            bs = stock.balance_sheet
            if bs is not None and not bs.empty:
                save_json(
                    "balance_sheet",
                    f"{safe_ticker}.json",
                    bs.reset_index().to_dict(orient="records")
                )
        except Exception as e:
            print(f"balance_sheet error {ticker}: {e}")

        # ---------------- CASHFLOW ----------------
        try:
            cf = stock.cashflow
            if cf is not None and not cf.empty:
                save_json(
                    "cashflow",
                    f"{safe_ticker}.json",
                    cf.reset_index().to_dict(orient="records")
                )
        except Exception as e:
            print(f"cashflow error {ticker}: {e}")

        # ---------------- INFO ----------------
        try:
            info = stock.info
            if info:
                save_json("info", f"{safe_ticker}.json", info)
        except Exception as e:
            print(f"info error {ticker}: {e}")

    except Exception as e:
        print(f"FAILED {ticker}: {e}")

    # rate limit safety
    time.sleep(1.5 + random.random())


# -----------------------------
# PARALLEL RUNNER
# -----------------------------
def run_parallel(companies, max_workers=4):

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(ingest_company, c) for c in companies]

        for f in as_completed(futures):
            try:
                f.result()
            except Exception as e:
                print(f"Worker failed: {e}")


# -----------------------------
# EXECUTE
# -----------------------------
run_parallel(companies, max_workers=4)

In [0]:
import yfinance as yf
import json
import time
import random
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# -----------------------------
# BASE PATH (Unity Catalog Volume)
# -----------------------------
BASE_PATH = "/Volumes/company-risk-intelligence-platform/bronze/raw_data/yfinance/"

SUBFOLDERS = [
    "stock",
    "news",
    "income_statement",
    "balance_sheet",
    "cashflow",
    "info"
]

# create folders
for folder in SUBFOLDERS:
    os.makedirs(BASE_PATH + folder, exist_ok=True)


# -----------------------------
# COMPANIES
# -----------------------------
companies = [
    {"name": "J SAINSBURY PLC", "ticker": "SBRY.L"},
    {"name": "JD SPORTS FASHION PLC", "ticker": "JD.L"},
    {"name": "OCADO GROUP PLC", "ticker": "OCDO.L"},
    {"name": "LLOYDS BANKING GROUP PLC", "ticker": "LLOY.L"},
    {"name": "NATWEST GROUP PLC", "ticker": "NWG.L"},
    {"name": "NATIONAL GRID PLC", "ticker": "NG.L"},
    {"name": "SSE PLC", "ticker": "SSE.L"},
    {"name": "DRAX GROUP PLC", "ticker": "DRX.L"},
    {"name": "BALFOUR BEATTY PLC", "ticker": "BBY.L"},
    {"name": "PERSIMMON PLC", "ticker": "PSN.L"},
    {"name": "EASYJET PLC", "ticker": "EZJ.L"},
    {"name": "INTERCONTINENTAL HOTELS GROUP PLC", "ticker": "IHG.L"},
    {"name": "HIKMA PHARMACEUTICALS PLC", "ticker": "HIK.L"},
    {"name": "BT GROUP PLC", "ticker": "BT-A.L"},
    {"name": "ITV PLC", "ticker": "ITV.L"},
    {"name": "JOHNSON MATTHEY PLC", "ticker": "JMAT.L"},
    {"name": "MELROSE INDUSTRIES PLC", "ticker": "MRO.L"},
    {"name": "AVIVA PLC", "ticker": "AV.L"},
    {"name": "ASTON MARTIN LAGONDA GLOBAL HOLDINGS PLC", "ticker": "AML.L"},
]


# -----------------------------
# SAFE DATAFRAME CONVERTER
# FIXES Timestamp → JSON issue
# -----------------------------
def safe_df_to_records(df):
    if df is None or df.empty:
        return None

    df = df.copy()

    # convert index/columns safely
    df.columns = [str(c) for c in df.columns]
    df = df.reset_index()

    # replace NaN with None
    df = df.where(pd.notnull(df), None)

    return df.to_dict(orient="records")


# -----------------------------
# SAVE FUNCTION
# -----------------------------
def save_json(folder, filename, data):
    if data is None:
        return

    path = f"{BASE_PATH}{folder}/{filename}"

    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)


# -----------------------------
# INGEST FUNCTION
# -----------------------------
def ingest_company(company):

    ticker = company["ticker"]
    name = company["name"]
    safe_ticker = ticker.replace(".", "_")

    print(f"\nProcessing {name} ({ticker})")

    try:
        stock = yf.Ticker(ticker)

        # ---------------- STOCK ----------------
        try:
            hist = stock.history(period="1y")
            save_json(
                "stock",
                f"{safe_ticker}.json",
                safe_df_to_records(hist)
            )
        except Exception as e:
            print(f"stock error {ticker}: {e}")

        # ---------------- NEWS ----------------
        try:
            save_json("news", f"{safe_ticker}.json", stock.news)
        except Exception as e:
            print(f"news error {ticker}: {e}")

        # ---------------- INCOME STATEMENT ----------------
        try:
            income = stock.financials
            save_json(
                "income_statement",
                f"{safe_ticker}.json",
                safe_df_to_records(income)
            )
        except Exception as e:
            print(f"income error {ticker}: {e}")

        # ---------------- BALANCE SHEET ----------------
        try:
            bs = stock.balance_sheet
            save_json(
                "balance_sheet",
                f"{safe_ticker}.json",
                safe_df_to_records(bs)
            )
        except Exception as e:
            print(f"balance sheet error {ticker}: {e}")

        # ---------------- CASHFLOW ----------------
        try:
            cf = stock.cashflow
            save_json(
                "cashflow",
                f"{safe_ticker}.json",
                safe_df_to_records(cf)
            )
        except Exception as e:
            print(f"cashflow error {ticker}: {e}")

        # ---------------- INFO ----------------
        try:
            save_json("info", f"{safe_ticker}.json", stock.info)
        except Exception as e:
            print(f"info error {ticker}: {e}")

    except Exception as e:
        print(f"FAILED {ticker}: {e}")

    time.sleep(1.5 + random.random())


# -----------------------------
# PARALLEL RUNNER
# -----------------------------
def run_parallel(companies, max_workers=4):

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(ingest_company, c) for c in companies]

        for f in as_completed(futures):
            try:
                f.result()
            except Exception as e:
                print(f"Worker failed: {e}")


# -----------------------------
# EXECUTE
# -----------------------------
run_parallel(companies, max_workers=4)

In [0]:
#folder-level validation
BASE_PATH = "/Volumes/company-risk-intelligence-platform/bronze/raw_data/yfinance/"

for folder in os.listdir(BASE_PATH):
    folder_path = os.path.join(BASE_PATH, folder)

    if os.path.isdir(folder_path):
        files = os.listdir(folder_path)
        print(f"{folder}: {len(files)} files")

In [0]:
#checking if each company exists in each folder
expected_tickers = [
    "SBRY_L", "JD_L", "OCDO_L", "LLOY_L", "NWG_L",
    "NG_L", "SSE_L", "DRX_L", "BBY_L", "PSN_L",
    "EZJ_L", "IHG_L", "HIK_L", "BT_A_L", "ITV_L",
    "JMAT_L", "MRO_L", "AV_L", "AML_L"
]

BASE_PATH = "/Volumes/company-risk-intelligence-platform/bronze/raw_data/yfinance/"

folders = os.listdir(BASE_PATH)

for folder in folders:
    folder_path = os.path.join(BASE_PATH, folder)

    if os.path.isdir(folder_path):
        existing = set([f.replace(".json", "") for f in os.listdir(folder_path)])

        missing = set(expected_tickers) - existing

        print(f"\n{folder}")
        print(f"Missing: {len(missing)}")
        if missing:
            print(missing)

In [0]:
#checking file intergrity(empty or broken json)
def validate_json_files(folder_path):
    bad_files = []

    for f in os.listdir(folder_path):
        path = os.path.join(folder_path, f)

        try:
            with open(path, "r") as file:
                data = json.load(file)

            if data is None or data == {} or data == []:
                bad_files.append(f)

        except Exception:
            bad_files.append(f)

    return bad_files


BASE_PATH = "/Volumes/company-risk-intelligence-platform/bronze/raw_data/yfinance/"

for folder in os.listdir(BASE_PATH):
    folder_path = os.path.join(BASE_PATH, folder)

    if os.path.isdir(folder_path):
        bad = validate_json_files(folder_path)

        print(f"{folder}: {len(bad)} bad files")
        if bad:
            print(bad[:5])

In [0]:
import json

file_path = "/Volumes/company-risk-intelligence-platform/bronze/raw_data/yfinance/stock/SBRY_L.json"

with open(file_path, "r") as f:
    data = json.load(f)

print(type(data))
print(len(data) if isinstance(data, list) else "not list")
print(data[:2])

In [0]:
import yfinance as yf
import json
import time
import random
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# -----------------------------
# BASE PATH (Unity Catalog Volume)
# -----------------------------
BASE_PATH = "/Volumes/company-risk-intelligence-platform/bronze/raw_data/yfinance/"

SUBFOLDERS = [
    "stock",
    "news",
    "income_statement",
    "balance_sheet",
    "cashflow",
    "info"
]

for folder in SUBFOLDERS:
    os.makedirs(BASE_PATH + folder, exist_ok=True)


# -----------------------------
# COMPANIES
# -----------------------------
companies = [
    {"name": "J SAINSBURY PLC", "ticker": "SBRY.L"},
    {"name": "JD SPORTS FASHION PLC", "ticker": "JD.L"},
    {"name": "OCADO GROUP PLC", "ticker": "OCDO.L"},
    {"name": "LLOYDS BANKING GROUP PLC", "ticker": "LLOY.L"},
    {"name": "NATWEST GROUP PLC", "ticker": "NWG.L"},
    {"name": "NATIONAL GRID PLC", "ticker": "NG.L"},
    {"name": "SSE PLC", "ticker": "SSE.L"},
    {"name": "DRAX GROUP PLC", "ticker": "DRX.L"},
    {"name": "BALFOUR BEATTY PLC", "ticker": "BBY.L"},
    {"name": "PERSIMMON PLC", "ticker": "PSN.L"},
    {"name": "EASYJET PLC", "ticker": "EZJ.L"},
    {"name": "INTERCONTINENTAL HOTELS GROUP PLC", "ticker": "IHG.L"},
    {"name": "HIKMA PHARMACEUTICALS PLC", "ticker": "HIK.L"},
    {"name": "BT GROUP PLC", "ticker": "BT-A.L"},   # FIXED HERE
    {"name": "ITV PLC", "ticker": "ITV.L"},
    {"name": "JOHNSON MATTHEY PLC", "ticker": "JMAT.L"},
    {"name": "MELROSE INDUSTRIES PLC", "ticker": "MRO.L"},
    {"name": "AVIVA PLC", "ticker": "AV.L"},
    {"name": "ASTON MARTIN LAGONDA GLOBAL HOLDINGS PLC", "ticker": "AML.L"},
]


# -----------------------------
# SAFE DF CONVERTER
# -----------------------------
def safe_df_to_records(df):
    if df is None or df.empty:
        return None

    df = df.copy()
    df.columns = [str(c) for c in df.columns]
    df = df.reset_index()
    df = df.where(pd.notnull(df), None)

    return df.to_dict(orient="records")


# -----------------------------
# SAVE FUNCTION
# -----------------------------
def save_json(folder, filename, data):
    if data is None:
        return

    path = f"{BASE_PATH}{folder}/{filename}"

    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)


# -----------------------------
# INGEST FUNCTION
# -----------------------------
def ingest_company(company):

    ticker = company["ticker"]
    name = company["name"]

    # FIXED: handles both '.' and '-'
    safe_ticker = ticker.replace(".", "_").replace("-", "_")

    print(f"\nProcessing {name} ({ticker})")

    try:
        stock = yf.Ticker(ticker)

        # STOCK
        try:
            hist = stock.history(period="1y")
            save_json("stock", f"{safe_ticker}.json", safe_df_to_records(hist))
        except Exception as e:
            print(f"stock error {ticker}: {e}")

        # NEWS
        try:
            save_json("news", f"{safe_ticker}.json", stock.news)
        except Exception as e:
            print(f"news error {ticker}: {e}")

        # INCOME STATEMENT
        try:
            income = stock.financials
            save_json("income_statement", f"{safe_ticker}.json", safe_df_to_records(income))
        except Exception as e:
            print(f"income error {ticker}: {e}")

        # BALANCE SHEET
        try:
            bs = stock.balance_sheet
            save_json("balance_sheet", f"{safe_ticker}.json", safe_df_to_records(bs))
        except Exception as e:
            print(f"balance sheet error {ticker}: {e}")

        # CASHFLOW
        try:
            cf = stock.cashflow
            save_json("cashflow", f"{safe_ticker}.json", safe_df_to_records(cf))
        except Exception as e:
            print(f"cashflow error {ticker}: {e}")

        # INFO
        try:
            save_json("info", f"{safe_ticker}.json", stock.info)
        except Exception as e:
            print(f"info error {ticker}: {e}")

    except Exception as e:
        print(f"FAILED {ticker}: {e}")

    time.sleep(1.5 + random.random())


# -----------------------------
# PARALLEL RUNNER
# -----------------------------
def run_parallel(companies, max_workers=4):

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(ingest_company, c) for c in companies]

        for f in as_completed(futures):
            try:
                f.result()
            except Exception as e:
                print(f"Worker failed: {e}")


# -----------------------------
# RUN
# -----------------------------
run_parallel(companies, max_workers=4)

In [0]:
import os
import json
import pandas as pd

BASE_PATH = "/Volumes/company-risk-intelligence-platform/bronze/raw_data/yfinance/"

expected_tickers = [
    "SBRY_L", "JD_L", "OCDO_L", "LLOY_L", "NWG_L",
    "NG_L", "SSE_L", "DRX_L", "BBY_L", "PSN_L",
    "EZJ_L", "IHG_L", "HIK_L", "BT_A_L", "ITV_L",
    "JMAT_L", "MRO_L", "AV_L", "AML_L"
]

expected_count = len(expected_tickers)

print("\n==============================")
print("🔍 INGESTION HEALTH CHECK")
print("==============================\n")


# -----------------------------
# 1. Folder-level completeness
# -----------------------------
print("📁 1. Folder-level completeness\n")

for folder in os.listdir(BASE_PATH):
    folder_path = os.path.join(BASE_PATH, folder)

    if os.path.isdir(folder_path):
        files = os.listdir(folder_path)
        print(f"{folder}: {len(files)} files")

        if len(files) != expected_count:
            print(f"⚠️ WARNING: expected {expected_count}, got {len(files)}")


# -----------------------------
# 2. Missing tickers per folder
# -----------------------------
print("\n📊 2. Missing tickers per folder\n")

expected_set = set(expected_tickers)

for folder in os.listdir(BASE_PATH):
    folder_path = os.path.join(BASE_PATH, folder)

    if os.path.isdir(folder_path):
        existing = set([f.replace(".json", "") for f in os.listdir(folder_path)])
        missing = expected_set - existing

        print(f"{folder} → Missing: {len(missing)}")
        if missing:
            print(missing)


# -----------------------------
# 3. JSON integrity check
# -----------------------------
print("\n🧪 3. JSON integrity check\n")

def validate_folder(path):
    bad = []

    for f in os.listdir(path):
        fp = os.path.join(path, f)

        try:
            with open(fp, "r") as file:
                data = json.load(file)

            if data is None:
                bad.append(f)
            elif isinstance(data, list) and len(data) == 0:
                bad.append(f)
            elif isinstance(data, dict) and len(data) == 0:
                bad.append(f)

        except Exception:
            bad.append(f)

    return bad


for folder in os.listdir(BASE_PATH):
    folder_path = os.path.join(BASE_PATH, folder)

    if os.path.isdir(folder_path):
        bad = validate_folder(folder_path)
        print(f"{folder}: {len(bad)} bad files")
        if bad:
            print(bad[:5])


# -----------------------------
# 4. Sample data check
# -----------------------------
print("\n🔎 4. Sample data check (stock)\n")

sample_file = f"{BASE_PATH}/stock/SBRY_L.json"

with open(sample_file, "r") as f:
    data = json.load(f)

print("Type:", type(data))
print("Rows:", len(data))
print("Sample:", data[:2])


# -----------------------------
# 5. Overall health score
# -----------------------------
print("\n📈 5. Overall health summary\n")

for folder in os.listdir(BASE_PATH):
    folder_path = os.path.join(BASE_PATH, folder)

    if os.path.isdir(folder_path):
        files = set([f.replace(".json", "") for f in os.listdir(folder_path)])

        completeness = (len(files) / expected_count) * 100
        missing = expected_set - files

        print(f"{folder}")
        print(f"  Completeness: {completeness:.1f}%")
        print(f"  Missing: {len(missing)}")

print("\n==============================")
print("✅ VALIDATION COMPLETE")
print("==============================")